# Research Agent — Interactive Notebook

Run a full multi-source research workflow using the graph-based pipeline.

This notebook runs **entirely with mock adapters** — no API keys required.

In [1]:
from __future__ import annotations

import sys
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env")

from research_agent.config import settings
from research_agent.logging import setup_logging
setup_logging(debug=settings.debug)

## 1. Run the full research graph

In [ ]:
from research_agent.graph.state import ResearchState
from research_agent.graph.workflow import build_graph

graph = build_graph(checkpointer=True)
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)

initial = ResearchState(
    question="What are the key developments in AI agent safety for 2026?",
    max_results_per_source=3,
    raw_search_results={},
    include_sources=["news", "web", "reddit", "wikipedia"],
)

result = await graph.ainvoke(
    initial,
    {"configurable": {"thread_id": "notebook-demo"}},
)

status = result.get("status", "unknown")
evidence_count = len(result.get("extracted_evidence", []))
warnings = result.get("warnings", [])

print(f"Status: {status}")
print(f"Evidence items: {evidence_count}")
if warnings:
    print(f"Warnings ({len(warnings)}):")
    for w in warnings:
        print(f"  - {w}")

{"question": "What are the key developments in AI agent safety for 2026?", "event": "planning_queries", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:01:39.282469Z"}
HTTP Request: POST http://host.docker.internal:11434/api/chat "HTTP/1.1 200 OK"
{"source": "news", "count": 0, "event": "source_search_complete", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:01:44.520129Z"}
{"source": "reddit", "count": 0, "event": "source_search_complete", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:01:44.523379Z"}
HTTP Request: GET https://hn.algolia.com/api/v1/search?query=People+are+saying+what+the+key+developments+in+AI+security+might+look+like+by+2026+on+Twitter+and+Reddit&hitsPerPage=5&tags=story "HTTP/1.1 200 OK"
{"source": "social", "count": 0, "event": "source_search_complete", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:01:44.961942Z"}
HTTP Request: GET https://en.wikipedia.org/w/api.

## 2. Inspect the query plan

In [3]:
from rich import print as rprint

plan = result.get("query_plan")
if plan:
    rprint(plan)
else:
    print("No query plan was generated.")

QueryPlan(
    original_question='What are the key developments in AI agent safety for 2026?',
    research_objective="Identify significant advancements and trends expected to influence artificial intelligence 
(AI) agents' security measures by mid-2026.",
    time_sensitivity='recent',
    constraints=['Future predictions', 'Expert analysis'],
    entities=['Artificial Intelligence Safety', 'AI Agents', 'Security Measures', 'Advancements', 'Trends'],
    search_plan=SearchPlan(
        news=[
            'Key developments in AI safety predicted for mid-2026 - News headline search',
            "Experts' forecast on the future of AI agent security by end of 2023"
        ],
        web=[
            'Future trends and advancements in artificial intelligence (AI) agents’ safety to be expected by June 
2026: In-depth analysis article',
            'How will AI agent safety evolve? Expert predictions for mid-2026 - Web guide/document search'
        ],
        social=[
            'People are saying what the key developments in AI security might look like by 2026 on Twitter and 
Reddit'
        ],
        reddit=[
            'Reddit thinks about how artificial intelligence agents will be safer by June 2026',
            'AI safety predictions for mid-2026: Community discussion thread search'
        ],
        wikipedia=[
            'Artificial Intelligence Safety - Overview of current trends as a reference point before looking into 
future developments'
        ]
    ),
    success_criteria=[
        'Expert analysis and forecasts on AI agent security advancements by the end of 2023',
        'In-depth articles discussing expected safety measures for artificial intelligence agents in mid-2026',
        'Community discussions reflecting public opinion or predictions about upcoming trends in AI safety'
    ],
    risks=[
        'Predictions may be speculative and not come to fruition as anticipated.',
        'Information could become outdated quickly due to the nature of technological advancements.'
    ]
)

## 3. View extracted evidence

In [4]:
evidence = result.get("extracted_evidence", [])

print(f"Extracted {len(evidence)} evidence items:\n")
for i, ev in enumerate(evidence):
    print(f"--- Evidence {i + 1} ---")
    print(f"  Source type: {ev.source_type}")
    print(f"  Title:       {ev.title}")
    print(f"  URL:         {ev.url}")
    print(f"  Publisher:   {ev.publisher_or_platform}")
    print(f"  Summary:     {ev.summary[:200] if ev.summary else '(none)'}")
    if ev.key_points:
        print(f"  Key points:  {len(ev.key_points)}")
        for kp in ev.key_points[:3]:
            print(f"    - {kp}")
    if ev.claims:
        print(f"  Claims:      {len(ev.claims)}")
        for c in ev.claims[:3]:
            print(f"    - [{c.confidence:.0%}] {c.claim[:120]}")
    if ev.limitations:
        print(f"  Limitations: {'; '.join(ev.limitations)}")
    print()

Extracted 6 evidence items:

--- Evidence 1 ---
  Source type: web
  Title:       International AI Safety Report 2026
  URL:         https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026
  Publisher:   
  Summary:     International AI Safety Report 2026
                The second International AI Safety Report, published in February 2026, is the next iteration of the comprehensive review of latest scientific resear

--- Evidence 2 ---
  Source type: web
  Title:       State of AI Agent Security Report
  URL:         https://www.gravitee.io/state-of-ai-agent-security
  Publisher:   
  Summary:     The State of AI Agent Security 2026
Updated April 2026. Our second survey of 750 senior technology leaders across the UK and USA reveals that the enterprise AI agent estate has doubled in four months,

--- Evidence 3 ---
  Source type: web
  Title:       AI Agent Security 2026: Google's Forecast and How to Fix the Gaps
  URL:         https://agatsoftware.com/a

## 4. Source reliability scoring

In [5]:
from research_agent.analysis.reliability import score_source_reliability

print(f"{'Source':<30} {'Reliability':<12} {'Type':<10}")
print("-" * 54)
for ev in evidence:
    rel = score_source_reliability(ev)
    print(f"{ev.title[:30]:30} {rel:<12.2f} {ev.source_type:<10}")

Source                         Reliability  Type      
------------------------------------------------------
International AI Safety Report 0.50         web       
State of AI Agent Security Rep 0.50         web       
AI Agent Security 2026: Google 0.50         web       
5 Predictions for AI Agent Sec 0.50         web       
Agentic AI Takes Over — 11 Sho 0.50         web       
Expert Predictions on What's a 0.50         web       


## 5. Cross-source synthesis

In [6]:
synthesis = result.get("synthesis")
if synthesis:
    if synthesis.summary:
        print("Summary:")
        print(f"  {synthesis.summary}\n")

    if synthesis.major_themes:
        print(f"Major themes ({len(synthesis.major_themes)}):")
        for t in synthesis.major_themes:
            print(f"  - {t}")

    if synthesis.repeated_claims:
        print(f"\nCorroborated claims ({len(synthesis.repeated_claims)}):")
        for c in synthesis.repeated_claims:
            print(f"  - {c}")

    if synthesis.contradictions:
        print(f"\nContradictions ({len(synthesis.contradictions)}):")
        for c in synthesis.contradictions:
            print(f"  - {c}")

    if synthesis.well_supported:
        print(f"\nWell-supported ({len(synthesis.well_supported)}):")
        for c in synthesis.well_supported:
            print(f"  - {c}")

    if synthesis.weakly_supported:
        print(f"\nWeakly supported ({len(synthesis.weakly_supported)}):")
        for c in synthesis.weakly_supported:
            print(f"  - {c}")

    if synthesis.requires_more_research:
        print(f"\nRequires more research ({len(synthesis.requires_more_research)}):")
        for r in synthesis.requires_more_research:
            print(f"  - {r}")
else:
    print("No synthesis was produced.")

## 6. Render the final markdown report

In [7]:
from IPython.display import Markdown, display

report = result.get("report_markdown", "")
display(Markdown(report))

# Research Report: What are the key developments in AI agent safety for 2026?

*Generated: 2026-07-16 16:03 UTC*  

*Evidence items: 6*  


## Executive Summary

See key findings below.

## Key Findings

## Evidence Table

| # | Source Type | Title | Reliability | Key Claims |
|---|------------|-------|-------------|------------|
| 1 | web | [International AI Safety Report 2026](https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026) | 0.50 |  |
| 2 | web | [State of AI Agent Security Report](https://www.gravitee.io/state-of-ai-agent-security) | 0.50 |  |
| 3 | web | [AI Agent Security 2026: Google's Forecast and How ](https://agatsoftware.com/ai-agent-security-2026-google-forecast) | 0.50 |  |
| 4 | web | [5 Predictions for AI Agent Security in 2026](https://neuraltrust.ai/blog/5-predictions-for-ai-agent-security-in-2026) | 0.50 |  |
| 5 | web | [Agentic AI Takes Over — 11 Shocking 2026 Predictio](https://www.forbes.com/sites/markminevich/2025/12/31/agentic-ai-takes-over-11-shocking-2026-predictions) | 0.50 |  |
| 6 | web | [Expert Predictions on What's at Stake in AI Policy](https://www.techpolicy.press/expert-predictions-on-whats-at-stake-in-ai-policy-in-2026) | 0.50 |  |

## Source Reliability Notes

| Source | Count | Avg Reliability | Notes |
|--------|-------|-----------------|-------|
| web | 6 | 0.50 | Varies by domain authority |

## Per-Source Summaries

### 1. International AI Safety Report 2026
- **Source**: [https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026](https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: International AI Safety Report 2026
                The second International AI Safety Report, published in February 2026, is the next iteration of the comprehensive review of latest scientific research on the capabilities and risks of general-purpose AI systems. Led by Turing Award winner Yoshua Bengio and authored by over 100 AI experts, the report is backed by over 30 countries and international organisations. It represents the largest global collaboration on AI safety to date. 
Translated ve

### 2. State of AI Agent Security Report
- **Source**: [https://www.gravitee.io/state-of-ai-agent-security](https://www.gravitee.io/state-of-ai-agent-security)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: The State of AI Agent Security 2026
Updated April 2026. Our second survey of 750 senior technology leaders across the UK and USA reveals that the enterprise AI agent estate has doubled in four months, while security coverage has barely moved.
Getting Started
Last updated on: April, 2026 | Published: June 15th, 2026 | Author: Jorge Ruiz
Executive Summary: 
The confidence-reality gap is widening
AI agent fleets have roughly doubled since December 2025. Confidence in security has risen. But monitor

### 3. AI Agent Security 2026: Google's Forecast and How to Fix the Gaps
- **Source**: [https://agatsoftware.com/ai-agent-security-2026-google-forecast](https://agatsoftware.com/ai-agent-security-2026-google-forecast)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: Google’s 2026 Cybersecurity Forecast Named Your Biggest AI Risks — Here’s How to Actually Fix Them
Introduction
AI agent security in 2026 has become one of the most urgent priorities on every CISO’s desk — and Google’s latest Cybersecurity Forecast explains exactly why. This post breaks down what Google’s Cybersecurity Forecast 2026 means for AI agent security in 2026 and what your team needs to do about it.
The 2026 edition lands hard on AI. Not in the abstract, futuristic sense that security h

### 4. 5 Predictions for AI Agent Security in 2026
- **Source**: [https://neuraltrust.ai/blog/5-predictions-for-ai-agent-security-in-2026](https://neuraltrust.ai/blog/5-predictions-for-ai-agent-security-in-2026)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: TL;DR Autonomous AI agents are rapidly being deployed, but security is dangerously lagging (72% adoption vs. 29% comprehensive security), based on our latest global CISO survey.
We predict five critical threats will dominate the near future: Indirect Prompt Injection (IPI) will become the primary attack vector, agentic browsers will turn the web into a weapon, the Model Context Protocol (MCP) will be the new high-value target, and Shadow AI will drive massive data leakage. Securing the autonomou

### 5. Agentic AI Takes Over — 11 Shocking 2026 Predictions
- **Source**: [https://www.forbes.com/sites/markminevich/2025/12/31/agentic-ai-takes-over-11-shocking-2026-predictions](https://www.forbes.com/sites/markminevich/2025/12/31/agentic-ai-takes-over-11-shocking-2026-predictions)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: As 2025 comes to a close, it has become clear to me that we are not simply watching a global technological change; we are, in fact, living within it. Artificial intelligence has become the force behind nearly everything. It is reshaping how companies operate, how workers perform and how economies compete. But what comes next?
As we look toward 2026, we must realize that AI isn’t a layer we add to systems, it’s becoming the infrastructure itself.
In 2026, AI will move with us as a constant co-wor

### 6. Expert Predictions on What's at Stake in AI Policy in 2026
- **Source**: [https://www.techpolicy.press/expert-predictions-on-whats-at-stake-in-ai-policy-in-2026](https://www.techpolicy.press/expert-predictions-on-whats-at-stake-in-ai-policy-in-2026)
- **Type**: web
- **Publisher**: 
- **Reliability**: 0.50
- **Summary**: Expert Predictions on What’s at Stake in AI Policy in 2026
J.B. Branch, Ilana Beller / Jan 6, 2026J.B. Branch is the Big Tech accountability advocate for Public Citizen’s Congress Watch division, and Ilana Beller leads Public Citizen’s state legislative work relating to artificial intelligence.
For years, debates over the regulation of artificial intelligence required a degree of speculation about its potential harms. But even as the technology continues to evolve, it is clear that by the end of

## Recommendations for Further Research


## Appendix

- **Research question**: What are the key developments in AI agent safety for 2026?
- **Time sensitivity**: recent
- **Entities identified**: Artificial Intelligence Safety, AI Agents, Security Measures, Advancements, Trends
- **Constraints**: Future predictions; Expert analysis


## 7. Export to JSON

In [8]:
from research_agent.report.json import generate_json_report

json_report = generate_json_report(
    question=result.get("question", ""),
    query_plan=result.get("query_plan"),
    evidence=result.get("extracted_evidence", []),
    synthesis=result.get("synthesis"),
    warnings=result.get("warnings", []),
)
print(json_report[:2000] + "...")

{
  "report_type": "research_report",
  "generated_at": "2026-07-16T16:04:29.322268+00:00",
  "question": "What are the key developments in AI agent safety for 2026?",
  "evidence_count": 6,
  "warnings": [],
  "query_plan": {
    "original_question": "What are the key developments in AI agent safety for 2026?",
    "research_objective": "Identify significant advancements and trends expected to influence artificial intelligence (AI) agents' security measures by mid-2026.",
    "time_sensitivity": "recent",
    "constraints": [
      "Future predictions",
      "Expert analysis"
    ],
    "entities": [
      "Artificial Intelligence Safety",
      "AI Agents",
      "Security Measures",
      "Advancements",
      "Trends"
    ],
    "search_plan": {
      "news": [
        "Key developments in AI safety predicted for mid-2026 - News headline search",
        "Experts' forecast on the future of AI agent security by end of 2023"
      ],
      "web": [
        "Future trends and advance

## 8. Run with different questions

Change the question and re-run the full pipeline.

In [14]:

questions = [
    "What are the latest trends in quantum computing?",
    "What is the current state of renewable energy storage?",
]

for q in questions:
    print(f"\n{'=' * 60}")
    print(f"QUESTION: {q}")
    print('=' * 60)

    state = ResearchState(
        question=q,
        max_results_per_source=2,
        include_sources=["news", "web", "wikipedia"],
        raw_search_results={},
    )
    r = await graph.ainvoke(state, {"configurable": {"thread_id": f"batch-{q[:20]}"}})

    print(f"  Status: {r.get('status')}")
    print(f"  Evidence: {len(r.get('extracted_evidence', []))} items")


QUESTION: What are the latest trends in quantum computing?
{"question": "What are the latest trends in quantum computing?", "event": "planning_queries", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:58:04.233633Z"}


HTTP Request: POST http://host.docker.internal:11434/api/chat "HTTP/1.1 200 OK"
{"source": "news", "count": 0, "event": "source_search_complete", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:58:07.912721Z"}
{"source": "reddit", "count": 0, "event": "source_search_complete", "logger": "research-agent", "level": "info", "timestamp": "2026-07-16T16:58:07.919491Z"}
HTTP Request: GET https://hn.algolia.com/api/v1/search?query=Trending+discussions+related+to+%27quantum+computing%27+on+Twitter+using+hashtags+like+%23QuantumComputing+and+%23TechTrends&hitsPerPage=5&tags=story "HTTP/1.1 200 OK"
HTTP Request: GET https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Quantum+computing+-+Wikipedia+page+for+an+overview+of+current+concepts+and+historical+context&format=json&srlimit=5&srprop=snippet%7Ctitlesnippet "HTTP/1.1 403 Forbidden"
HTTP Request: GET https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=List+of+quantum+computers+as+per+W

## 9. Inspect raw search results (low-level)

In [15]:
raw = result.get("raw_search_results", {})
for source_type, results in raw.items():
    print(f"\n[{source_type.upper()}]")
    for r in results:
        print(f"  {r.title[:70]}")
        print(f"  URL: {r.url}")
        print(f"  Score: {r.score:.2f}  |  Publisher: {r.publisher}")
        print()


[NEWS]

[REDDIT]

[SOCIAL]

[WEB]
  International AI Safety Report 2026
  URL: https://internationalaisafetyreport.org/publication/international-ai-safety-report-2026
  Score: 0.93  |  Publisher: 

  State of AI Agent Security Report
  URL: https://www.gravitee.io/state-of-ai-agent-security
  Score: 0.90  |  Publisher: 

  AI Agent Security 2026: Google's Forecast and How to Fix the Gaps
  URL: https://agatsoftware.com/ai-agent-security-2026-google-forecast
  Score: 0.88  |  Publisher: 

  5 Predictions for AI Agent Security in 2026
  URL: https://neuraltrust.ai/blog/5-predictions-for-ai-agent-security-in-2026
  Score: 0.91  |  Publisher: 

  Agentic AI Takes Over — 11 Shocking 2026 Predictions
  URL: https://www.forbes.com/sites/markminevich/2025/12/31/agentic-ai-takes-over-11-shocking-2026-predictions
  Score: 0.89  |  Publisher: 

  Expert Predictions on What's at Stake in AI Policy in 2026
  URL: https://techpolicy.press/expert-predictions-on-whats-at-stake-in-ai-policy-in-2026
  

In [16]:
print("Done.")

Done.
